# Gate characterisation — what gate is this, really?

A *classical* truth table (AND, OR) cannot describe a quantum gate: AND/OR are
irreversible, quantum evolution is unitary and reversible. The two-qubit quantum
gates are **CNOT, CZ, SWAP, iSWAP** and their square roots.

So the right object is the **4×4 complex gate matrix** `M`, where

`M[i,j] = ⟨out_i | U | in_j⟩`   over the atomic basis `|gg⟩, |ge⟩, |eg⟩, |ee⟩`
(cavity in vacuum at start and end; atom 1 is the first symbol).

Two things a population table cannot tell you, and this one can:
- **is it unitary?** (`M†M = I`) — if not, it is not a gate at all
- **what are the phases?** — two different gates can share a population table

We run this **lossless** (Schrödinger evolution, no κ/γ), because a gate is defined
by its ideal action; loss is then a separate fidelity question.

In [ ]:
using QuantumOptics
using LinearAlgebra
using Plots
gr()

const n_max = 2
const g0    = 1.0
const w     = 1.0
const v_half = 4 / sqrt(pi) * g0 * w
const v_full = 2 / sqrt(pi) * g0 * w

b_cav = FockBasis(n_max)
b_at  = SpinBasis(1//2)
Ic, Ia = one(b_cav), one(b_at)

a   = destroy(b_cav) ⊗ Ia ⊗ Ia
s1m = Ic ⊗ sigmam(b_at) ⊗ Ia
s2m = Ic ⊗ Ia ⊗ sigmam(b_at)
sz1 = Ic ⊗ sigmaz(b_at) ⊗ Ia
sz2 = Ic ⊗ Ia ⊗ sigmaz(b_at)

num  = dagger(a) * a
exc1 = (sz1 + one(sz1)) / 2
exc2 = (sz2 + one(sz2)) / 2
Hc1 = dagger(a) * s1m + a * dagger(s1m)
Hc2 = dagger(a) * s2m + a * dagger(s2m)

gpulse(t, t0, v) = g0 * exp(-((v * (t - t0)) / w)^2)

labels   = ["gg", "ge", "eg", "ee"]
nexc     = [0, 1, 1, 2]                     # excitations in each basis state
ket_at(c) = c == 'e' ? spinup(b_at) : spindown(b_at)
full_ket(s) = fockstate(b_cav, 0) ⊗ ket_at(s[1]) ⊗ ket_at(s[2])

println("setup ok")

## Building the gate matrix

For each of the four inputs we evolve **unitarily** (`schroedinger_dynamic`) and
project the final state onto each of the four outputs.

One correction is needed. The term `Δ(exc1+exc2)` makes every excited state pick up
a trivial phase `e^{-iΔt}`. Since excitation number is conserved, that is a known
phase per excitation-number block, so we simply undo it. What remains is the real
gate action. (Any leftover *local* Z-phases are free in experiment — they are
absorbed into single-qubit calibration — so gates are compared up to those.)

In [ ]:
function gate_matrix(f, T; correct_Δ = 0.0)
    M = zeros(ComplexF64, 4, 4)
    tmax = last(T)
    for (j, si) in enumerate(labels)
        _, ψt = timeevolution.schroedinger_dynamic(T, full_ket(si), f)
        ψf = ψt[end]
        for (i, so) in enumerate(labels)
            M[i, j] = dagger(full_ket(so)) * ψf
        end
    end
    # undo the trivial per-excitation phase e^{-i Δ t n}
    for i in 1:4, j in 1:4
        M[i, j] *= exp(im * correct_Δ * tmax * nexc[i])
    end
    return M
end

# --- the sequential (resonant) protocol: atom1 half transit, atom2 full transit
function M_sequential(; Δ = 0.0, dt = 3.0, nsteps = 1500)
    t01 = 4 * (w / v_half); t02 = t01 + dt
    T = range(0, t02 + 4 * (w / v_full), length = nsteps)
    H0 = Δ * (exc1 + exc2)
    f(t, ψ) = H0 + gpulse(t, t01, v_half) * Hc1 + gpulse(t, t02, v_full) * Hc2
    gate_matrix(f, T; correct_Δ = Δ)
end

# --- the dispersive protocol: both atoms together, far detuned, slow transit
function M_dispersive(Δ; v = nothing, nsteps = 2500)
    v === nothing && (v = g0^2 * w * sqrt(2/pi) / Δ)
    t0 = 4 * (w / v)
    T = range(0, 8 * (w / v), length = nsteps)
    H0 = Δ * (exc1 + exc2)
    f(t, ψ) = H0 + gpulse(t, t0, v) * (Hc1 + Hc2)
    gate_matrix(f, T; correct_Δ = Δ), v
end

println("gate_matrix ready")

## Reporting helpers

Unitarity is the pass/fail test; the population table is `|M|²`.

In [ ]:
function show_gate(M; name = "gate")
    println("=== ", name, " ===")
    println("populations |M[i,j]|^2   (columns = input, rows = output)")
    println("        in:   gg      ge      eg      ee")
    for i in 1:4
        print("  out ", labels[i], ":  ")
        for j in 1:4
            print(lpad(string(round(abs2(M[i,j]), digits=3)), 7), " ")
        end
        println()
    end
    dev = opnorm(M' * M - I)
    println("unitarity  ||M†M - I|| = ", round(dev, digits=4),
            dev < 0.05 ? "   -> UNITARY (valid gate)" : "   -> NOT unitary (leaks out of the atomic subspace)")
    println("leakage (1 - column norm): ",
            [round(1 - sum(abs2, M[:,j]), digits=3) for j in 1:4])
    return dev
end

# gate fidelity against a target, maximised over local Z phases on each qubit
function gate_fidelity_localZ(M, U; ngrid = 60)
    best = 0.0
    for φ1 in range(0, 2π, length = ngrid), φ2 in range(0, 2π, length = ngrid)
        D = Diagonal([1, exp(im*φ2), exp(im*φ1), exp(im*(φ1+φ2))])
        Mc = D * M
        F = abs2(tr(U' * Mc)) / 16
        F > best && (best = F)
    end
    return best
end

iSWAP  = ComplexF64[1 0 0 0; 0 0 im 0; 0 im 0 0; 0 0 0 1]
sqiSWAP = ComplexF64[1 0 0 0;
                     0 1/sqrt(2) im/sqrt(2) 0;
                     0 im/sqrt(2) 1/sqrt(2) 0;
                     0 0 0 1]
SWAPg  = ComplexF64[1 0 0 0; 0 0 1 0; 0 1 0 0; 0 0 0 1]

println("helpers ready")

## 1. The sequential resonant protocol

Expect this to **fail** the unitarity test: the `ge` and `ee` inputs strand a photon
in the cavity, so those columns lose norm and `M†M ≠ I`.

In [ ]:
Mseq = M_sequential(Δ = 0.0, dt = 3.0)
show_gate(Mseq; name = "sequential resonant protocol")

## 2. The dispersive protocol

Here the exchange is mediated by a **virtual** photon: excitation-conserving,
symmetric between the two atoms, cavity stays empty. Expect `M†M ≈ I` and a
population table of iSWAP/SWAP shape.

If unitarity is poor, raise Δ. If nothing happens (M ≈ identity), the transit is too
fast — pass a smaller `v` explicitly.

In [ ]:
Mdis, vdis = M_dispersive(8.0)
println("transit velocity used: ", round(vdis, digits=4), "\n")
show_gate(Mdis; name = "dispersive protocol (Δ = 8)")

println()
println("fidelity to iSWAP    (up to local Z): ", round(gate_fidelity_localZ(Mdis, iSWAP), digits=3))
println("fidelity to sqrt-iSWAP (up to local Z): ", round(gate_fidelity_localZ(Mdis, sqiSWAP), digits=3))
println("fidelity to SWAP     (up to local Z): ", round(gate_fidelity_localZ(Mdis, SWAPg), digits=3))

## 3. Tuning the exchange angle

The exchange angle is set by how long the atoms overlap in the cavity, i.e. by the
transit velocity. Sweeping `v` walks you along the iSWAP family:

- **half exchange → √iSWAP**: the *entangling* gate (this is the one that makes your
  Bell state, and it is universal together with single-qubit rotations)
- **full exchange → iSWAP**: swaps the excitation between the atoms

The plot shows the population transferred `|M[eg→ge]|²` against velocity: pick the
velocity where it crosses **0.5** for √iSWAP, and **1.0** for iSWAP.

In [ ]:
Δfix = 8.0
vscan = range(0.4, 2.5, length = 18) .* (g0^2 * w * sqrt(2/pi) / Δfix)
swapped, unit_dev = Float64[], Float64[]
for v in vscan
    M, _ = M_dispersive(Δfix; v = v, nsteps = 2000)
    push!(swapped, abs2(M[2,3]))              # |eg> -> |ge>
    push!(unit_dev, opnorm(M'*M - I))
end

plot(vscan, swapped, m=:circle, lw=2, label="|M[eg→ge]|²",
     xlabel="transit velocity v", ylabel="value",
     title="Walking the iSWAP family (Δ = 8)")
plot!(vscan, unit_dev, m=:square, lw=2, label="unitarity error ||M†M−I||")
hline!([0.5], ls=:dash, lc=:black, label="√iSWAP (half exchange)")
hline!([1.0], ls=:dot,  lc=:black, label="iSWAP (full exchange)")

In [ ]:
Msq, _ = M_dispersive(8.0; v = 0.195)
show_gate(Msq; name = "dispersive, half exchange")
println("fidelity to sqrt-iSWAP: ", round(gate_fidelity_localZ(Msq, sqiSWAP), digits=3))

## 4. Same idea for the two **cavities**

Your cavity–cavity gate can be characterised exactly the same way — the only change
is the encoding. A cavity qubit is the Fock space `|0⟩ ≡ logical 0`, `|1⟩ ≡ logical 1`,
so the basis becomes `|0_A 0_B⟩, |0_A 1_B⟩, |1_A 0_B⟩, |1_A 1_B⟩` and the atom is the
mediator that must **exit disentangled** for the operation to be a gate on the cavities.

Same two tests apply: is `M` unitary (does the atom leave no trace?), and what gate
does it match. Build `full_ket` from `fockstate(bA, ·) ⊗ atom ⊗ fockstate(bB, ·)`,
require the atom to end in `|g⟩`, and reuse `gate_matrix` unchanged.

In [ ]:
function gate_fid_lossy(Δ, κ, γ; v = 0.0997, nsteps = 2000)
    t0 = 4*(w/v); T = range(0, 8*(w/v), length = nsteps)
    H0 = Δ*(exc1 + exc2)
    Jl = AbstractOperator[]
    κ > 0 && push!(Jl, sqrt(κ)*a)
    γ > 0 && push!(Jl, sqrt(γ)*s1m); γ > 0 && push!(Jl, sqrt(γ)*s2m)
    Jd = dagger.(Jl)
    f(t, ρ) = (H0 + gpulse(t, t0, v)*(Hc1 + Hc2), Jl, Jd)
    ideal = ["gg", "eg", "ge", "ee"]          # iSWAP action, in populations
    Fs = Float64[]
    for (j, si) in enumerate(labels)
        _, ρt = timeevolution.master_dynamic(T, full_ket(si), f)
        ψid = full_ket(ideal[j])
        push!(Fs, real(dagger(ψid) * (ρt[end] * ψid)))
    end
    return sum(Fs)/4
end

kaps = range(0.0, 0.2, length = 12)
Fk = [gate_fid_lossy(8.0, κ, 0.02) for κ in kaps]
plot(kaps, Fk, m=:circle, lw=2, legend=false,
     xlabel="cavity loss κ / g0", ylabel="average gate fidelity",
     title="Dispersive iSWAP under loss (Δ = 8)")

In [ ]:
function gate_fid_Δ(Δ, κ, γ; nsteps = 2000)
    v  = g0^2 * w * sqrt(2/pi) / Δ          # v calibrated for full iSWAP at this Δ
    t0 = 4*(w/v); T = range(0, 8*(w/v), length = nsteps)
    H0 = Δ*(exc1 + exc2)
    Jl = AbstractOperator[]
    κ > 0 && push!(Jl, sqrt(κ)*a)
    γ > 0 && push!(Jl, sqrt(γ)*s1m); γ > 0 && push!(Jl, sqrt(γ)*s2m)
    Jd = dagger.(Jl)
    f(t, ρ) = (H0 + gpulse(t, t0, v)*(Hc1 + Hc2), Jl, Jd)
    ideal = ["gg", "eg", "ge", "ee"]
    F = 0.0
    for (j, si) in enumerate(labels)
        _, ρt = timeevolution.master_dynamic(T, full_ket(si), f)
        ψid = full_ket(ideal[j])
        F += real(dagger(ψid) * (ρt[end] * ψid))
    end
    return F/4
end

deltas = range(2.0, 16.0, length = 12)
F_lowk  = [gate_fid_Δ(Δ, 0.0,  0.005) for Δ in deltas]
F_highk = [gate_fid_Δ(Δ, 0.20, 0.005) for Δ in deltas]

plot(deltas, F_lowk,  m=:circle, lw=2, label="κ = 0",
     xlabel="detuning Δ / g0", ylabel="average gate fidelity",
     title="Optimal detuning: κ-protection vs gate duration")
plot!(deltas, F_highk, m=:square, lw=2, label="κ = 0.2")

In [ ]:
deltas2 = range(2.0, 6.0, length = 14)
for (κv, lab, mk) in [(0.0,"κ = 0",:circle), (0.05,"κ = 0.05",:diamond),
                      (0.20,"κ = 0.2",:square), (0.50,"κ = 0.5",:utriangle)]
    F = [gate_fid_Δ(Δ, κv, 0.005) for Δ in deltas2]
    κv == 0.0 ? plot(deltas2, F, m=mk, lw=2, label=lab,
                     xlabel="detuning Δ / g0", ylabel="average gate fidelity",
                     title="Optimal detuning shifts with cavity loss") :
                plot!(deltas2, F, m=mk, lw=2, label=lab)
end
current()

In [ ]:
for κv in [0.0, 0.05, 0.2, 0.5]
    F = [gate_fid_Δ(Δ, κv, 0.005) for Δ in deltas2]
    i = argmax(F)
    println("κ = ", κv, "  ->  optimal Δ ≈ ", round(deltas2[i], digits=2),
            "   F = ", round(F[i], digits=4))
end